# Day 2 - File Handling & Modules

> Part of Day 2: Functions & OOP (final topic of Day 2)

This notebook covers:
- File I/O (open, read, write, with statement)
- Working with JSON
- Working with CSV
- Modules and imports


## 1. File Handling - Introduction

- File handling means reading data from files and writing data to files using Python
- A file is a resource on disk (hard drive) that stores data permanently
- Why do we need it?
  - Variables and objects in Python only live in memory (RAM) - they disappear once the program ends
  - Files let us store data permanently, so it survives after the program stops
  - Used to save logs, configs, datasets, reports, user data, etc.
- Where is it used?
  - Reading datasets (CSV, JSON, text) for data science
  - Writing logs in production applications
  - Saving configuration settings
  - Exporting reports


## 2. Real-Life Analogy

- Think of a **notebook/register** in real life
  - `open()` = opening the register to a page
  - `read()` = reading what is already written
  - `write()` = writing new content into it
  - `close()` = closing the register so nobody else disturbs it while you are away
- If you forget to close the register (file), it might stay "open" and get damaged or locked - same happens with files in Python if not closed properly


## 3. Explanation

- Python interacts with files using a **file object**, created using the built-in `open()` function
- Every file operation needs 3 steps:
  1. Open the file
  2. Perform the operation (read/write/append)
  3. Close the file (release the resource)
- Instead of manually closing, Python gives the `with` statement which **closes the file automatically**, even if an error occurs in between
- File modes decide what you are allowed to do with the file:

| (explained as pointers, not table, per formatting rule) |


### File Modes (explained as pointers)

- `"r"`  -> Read mode (default). File must already exist. Error if file not found
- `"w"`  -> Write mode. Creates file if not present, **overwrites/erases existing content**
- `"a"`  -> Append mode. Creates file if not present, adds new content at the end, does not erase
- `"x"`  -> Exclusive creation. Fails if file already exists
- `"r+"` -> Read and write, file must exist
- `"w+"` -> Write and read, overwrites existing content
- `"a+"` -> Append and read
- `"rb"`, `"wb"`, `"ab"` -> Same as above but in **binary mode** (used for images, PDFs, etc.)

> Important: Text mode (`"r"`, `"w"`) treats content as string (`str`). Binary mode (`"rb"`, `"wb"`) treats content as bytes.


## 4. Syntax

```python
# Basic open (not recommended - must close manually)
file = open("filename.txt", "r")
content = file.read()
file.close()

# Recommended - using 'with' statement (context manager)
with open("filename.txt", "r") as file:
    content = file.read()
# file is automatically closed here, even if an exception occurs
```

- `open(file, mode, encoding)` keywords:
  - `file` -> path to the file (string)
  - `mode` -> `"r"`, `"w"`, `"a"`, etc. (explained above)
  - `encoding` -> usually `"utf-8"`, recommended to always specify explicitly
- `with` keyword:
  - Creates a **context manager**
  - Guarantees `file.close()` runs automatically when the block ends
  - This is the **Pythonic and best practice** way to handle files


In [ ]:
# Basic Example - Writing to a file and reading it back

# Step 1: Write some lines into a file
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("Hello, this is line 1.\n")
    f.write("This is line 2.\n")
    f.write("This is line 3.\n")

# Step 2: Read the entire file content
with open("sample.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)


**Line by line explanation:**
- `open("sample.txt", "w", ...)` -> opens (creates) `sample.txt` in write mode
- `f.write(...)` -> writes a string into the file; `\n` means new line
- The `with` block ends -> file is automatically closed and saved to disk
- Second `with` block opens the same file in read mode (`"r"`)
- `f.read()` -> reads the **entire file content** as a single string


In [ ]:
# Intermediate Example - Reading line by line and using readlines()

with open("sample.txt", "r", encoding="utf-8") as f:
    # readline() reads ONE line at a time
    first_line = f.readline()
    print("First line:", first_line.strip())

with open("sample.txt", "r", encoding="utf-8") as f:
    # readlines() reads ALL lines and returns a LIST of strings
    all_lines = f.readlines()
    print("All lines as list:", all_lines)

with open("sample.txt", "r", encoding="utf-8") as f:
    # Best practice: iterate directly over file object (memory efficient)
    for line in f:
        print("Line ->", line.strip())


**Line by line explanation:**
- `f.readline()` -> reads only the **next single line** from the file, includes `\n` at the end
- `.strip()` -> removes leading/trailing whitespace and newline characters
- `f.readlines()` -> reads **all lines** and returns them as a **list of strings**
- Iterating directly with `for line in f:` -> reads the file **one line at a time internally**, without loading the whole file into memory - most memory efficient for large files


In [ ]:
# Real-world Example - Appending logs to a file (like an application log file)

def log_message(message):
    with open("app_log.txt", "a", encoding="utf-8") as f:
        f.write(message + "\n")

log_message("Application started")
log_message("User logged in")
log_message("User performed an action")

# Reading back the log file
with open("app_log.txt", "r", encoding="utf-8") as f:
    print(f.read())


**Line by line explanation:**
- `log_message()` is a small reusable function that **appends** (`"a"` mode) a message to a log file
- Every call to `log_message()` adds a new line **without erasing previous logs**
- This is exactly how real applications write log files (e.g., server logs, error logs)
- Reading the file afterwards shows all logged messages in order


## 5. Internal Working

- When you call `open()`, the operating system allocates a **file descriptor** (a low-level handle/reference to the file)
- Python keeps this file descriptor inside the file object it returns
- Read/write operations go through **buffered I/O**:
  - Data is not written to disk instantly on every `.write()` call
  - Python (and the OS) keep data in a temporary **buffer** in memory
  - The buffer is flushed (actually written to disk) when:
    - The buffer becomes full, OR
    - `f.close()` is called, OR
    - The `with` block ends (which calls close automatically)
- This is why `with` is important - if the program crashes before `close()`, buffered data might never reach the disk
- `with` internally works using Python's **context manager protocol** (`__enter__` and `__exit__` methods), which we will study in more depth as part of OOP


## 6. Time & Space Complexity

- `f.read()` -> Time: O(n), Space: O(n) - where n is total file size (entire content loaded into memory)
- `f.readline()` -> Time: O(1) amortized per call for reading one line, Space: O(k) where k is the length of that line
- `f.readlines()` -> Time: O(n), Space: O(n) - loads entire file into memory as a list
- Iterating with `for line in f:` -> Time: O(n) total, Space: O(k) - only one line is kept in memory at a time (most space efficient)

> Interview tip: For very large files (GBs of data), always prefer `for line in f:` over `read()` or `readlines()` to avoid loading everything into memory at once.


## 7. Common Mistakes

- Forgetting to close the file when not using `with` -> can cause data loss or file locking issues
- Opening a file in `"w"` mode when you actually wanted `"a"` mode -> accidentally **erases existing content**
- Not specifying `encoding="utf-8"` -> can cause errors with special characters on different operating systems
- Using `f.read()` on huge files -> can consume too much memory and slow down or crash the program
- Trying to read a file that doesn't exist without handling the error -> raises `FileNotFoundError`
- Forgetting `\n` while writing multiple lines -> all content gets written on a single line


## 8. Best Practices

- Always use `with open(...) as f:` instead of manual `open()`/`close()`
- Always specify `encoding="utf-8"` explicitly for text files
- Use `for line in f:` for large files instead of `read()`/`readlines()`
- Use meaningful variable names (`f`, `file`, `log_file`) instead of single unrelated letters
- Combine file handling with `try-except` to gracefully handle missing files (covered in Error Handling)
- Use relative paths carefully - be aware of the current working directory


## 9. Working with JSON - Introduction

- JSON (JavaScript Object Notation) is a lightweight, text-based data format used to store and exchange data
- Why do we need it?
  - It is the **most common format** for APIs, config files, and data exchange between systems
  - It maps very naturally to Python dictionaries and lists
- Where is it used?
  - REST APIs (sending/receiving data between frontend and backend)
  - Configuration files
  - Storing structured data (like a mini-database) in a simple text file


## 10. Real-Life Analogy

- Think of JSON like a **standard form/application form**
  - Everyone across different countries/companies can understand the same form format
  - JSON is a **universal format** understood by almost every programming language
  - Python dict -> JSON object, Python list -> JSON array


## 11. Explanation

- Python's built-in `json` module lets us convert between:
  - **Python objects <-> JSON text**
- Four main functions:
  - `json.dump(obj, file)` -> writes a Python object as JSON **into a file**
  - `json.dumps(obj)` -> converts a Python object into a JSON **string** (in memory, not a file)
  - `json.load(file)` -> reads JSON **from a file** and converts it into a Python object
  - `json.loads(string)` -> converts a JSON **string** into a Python object

> Remember: the extra "s" means "string" - `dumps`/`loads` work with strings, `dump`/`load` work with files.


## 12. Syntax

```python
import json

# Writing Python object to a JSON file
with open("data.json", "w") as f:
    json.dump(python_object, f)

# Reading JSON file into Python object
with open("data.json", "r") as f:
    data = json.load(f)

# Convert Python object to JSON string
json_string = json.dumps(python_object)

# Convert JSON string to Python object
python_object = json.loads(json_string)
```


In [ ]:
# Basic Example - Converting between Python dict and JSON string

import json

student = {
    "name": "Sandipan",
    "age": 30,
    "skills": ["Python", "SQL", "Machine Learning"],
    "is_employed": True
}

# Python dict -> JSON string
json_string = json.dumps(student)
print("JSON string:", json_string)
print("Type:", type(json_string))

# JSON string -> Python dict
back_to_dict = json.loads(json_string)
print("Back to dict:", back_to_dict)
print("Type:", type(back_to_dict))


**Line by line explanation:**
- `student` is a normal Python dictionary with mixed data types (string, int, list, bool)
- `json.dumps(student)` -> converts the dictionary into a **JSON formatted string**
- Notice the output looks like a dict but is actually of type `str`
- `json.loads(json_string)` -> parses the JSON string back into a Python dictionary


In [ ]:
# Intermediate Example - Saving and loading JSON to/from a file, with formatting

import json

data = {
    "course": "Python for Data Science",
    "days": 5,
    "topics_covered": ["Fundamentals", "Functions & OOP"]
}

# Save to file with indentation for readability
with open("course_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

# Load back from file
with open("course_data.json", "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

print(loaded_data)
print(type(loaded_data))


**Line by line explanation:**
- `json.dump(data, f, indent=4)` -> writes `data` into the file `course_data.json`
- `indent=4` -> makes the JSON file **human-readable** with 4-space indentation (otherwise it is a single long line)
- `json.load(f)` -> reads the file content and converts it back into a Python dictionary


In [ ]:
# Real-world Example - Simulating a simple API response being processed

import json

# Imagine this string came from an API response
api_response = '''
{
    "status": "success",
    "user": {
        "id": 101,
        "name": "Sandipan Paul",
        "roles": ["admin", "editor"]
    }
}
'''

response_data = json.loads(api_response)

if response_data["status"] == "success":
    user = response_data["user"]
    print(f"Welcome, {user['name']}! Your roles are: {', '.join(user['roles'])}")


**Line by line explanation:**
- `api_response` simulates a raw JSON string, exactly how data arrives from a web API
- `json.loads(api_response)` -> parses it into a nested Python dictionary
- We then access nested keys (`response_data["user"]`) just like a normal dict
- This is exactly the pattern used in real-world API integrations


## 13. Internal Working (JSON)

- Internally, `json.dumps()` walks through the Python object recursively
- It maps Python types to JSON types as follows (pointer-style, not a table):
  - `dict` -> JSON object `{ }`
  - `list` / `tuple` -> JSON array `[ ]`
  - `str` -> JSON string
  - `int` / `float` -> JSON number
  - `True` / `False` -> JSON `true` / `false`
  - `None` -> JSON `null`
- `json.loads()` does the reverse mapping while parsing the text character by character (a process called **parsing/tokenizing**)


## 14. Time & Space Complexity (JSON)

- `json.dumps()` / `json.dump()` -> Time: O(n), Space: O(n), where n is the total number of elements/characters in the object
- `json.loads()` / `json.load()` -> Time: O(n), Space: O(n), since the entire JSON text must be parsed and converted into Python objects


## 15. Common Mistakes (JSON)

- Confusing `dump`/`load` (files) with `dumps`/`loads` (strings)
- Trying to `json.dumps()` a Python object that JSON does not support directly (e.g., a `set`, or a custom class object) -> raises `TypeError`
- Forgetting that JSON keys are always strings - even if you use integer keys in a Python dict, they get converted to strings in JSON
- Not handling `json.JSONDecodeError` when the input JSON string is malformed


## 16. Best Practices (JSON)

- Use `indent=4` when writing JSON files meant to be read by humans
- Always validate/handle errors when parsing JSON from an external/untrusted source (API, user input)
- Use `ensure_ascii=False` if you need to store non-English characters properly
- Prefer JSON over manually parsing text when exchanging structured data between systems


## 17. Working with CSV - Introduction

- CSV (Comma Separated Values) is a simple text format used to store **tabular data** (rows and columns)
- Why do we need it?
  - It is the most common format for spreadsheets, datasets, and data exports
  - Every row is a record, every comma-separated value is a column/field
- Where is it used?
  - Exporting/importing data from Excel, databases, data science datasets
  - Simple data storage without needing a full database


## 18. Real-Life Analogy

- Think of a CSV file like an **Excel sheet saved as plain text**
  - Each line = one row (like one row in Excel)
  - Each comma = boundary between columns (like moving to the next cell)
- `csv.reader` / `csv.writer` = like opening/using Excel row by row, but without any formatting or formulas


## 19. Explanation

- Python's built-in `csv` module makes it easy to read and write CSV files correctly
  - It automatically handles tricky cases like commas inside quoted text
- Main tools:
  - `csv.reader(file)` -> reads CSV rows as **lists**
  - `csv.writer(file)` -> writes rows as **lists**
  - `csv.DictReader(file)` -> reads CSV rows as **dictionaries** (using the header row as keys)
  - `csv.DictWriter(file)` -> writes rows from **dictionaries**, using specified column headers


## 20. Syntax

```python
import csv

# Writing rows (list of lists)
with open("data.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age"])       # header
    writer.writerow(["Sandipan", 30])      # data row

# Reading rows as lists
with open("data.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

# Writing rows using dictionaries
with open("data.csv", "w", newline="", encoding="utf-8") as f:
    fieldnames = ["name", "age"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({"name": "Sandipan", "age": 30})

# Reading rows as dictionaries
with open("data.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)
```

- `newline=""` -> **must** be used when opening a file for CSV writing on Windows, to prevent extra blank lines being inserted


In [ ]:
# Basic Example - Writing and reading a CSV file using csv.writer / csv.reader

import csv

# Writing rows
with open("students.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "course"])          # header row
    writer.writerow(["Sandipan", 30, "Data Science"])
    writer.writerow(["Riya", 25, "Web Development"])

# Reading rows
with open("students.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)


**Line by line explanation:**
- `csv.writer(f)` -> creates a writer object linked to the file
- `writer.writerow([...])` -> writes one row; each list item becomes one column value
- `csv.reader(f)` -> creates a reader object that iterates over the file
- Each `row` returned is a **plain Python list of strings** (note: `30` becomes `"30"` as string when read back)


In [ ]:
# Intermediate Example - Using DictReader and DictWriter for cleaner, labeled data

import csv

fieldnames = ["name", "age", "course"]

# Writing using dictionaries
with open("students2.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({"name": "Sandipan", "age": 30, "course": "Data Science"})
    writer.writerow({"name": "Riya", "age": 25, "course": "Web Development"})

# Reading using dictionaries
with open("students2.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["name"], "->", row["course"])


**Line by line explanation:**
- `csv.DictWriter(f, fieldnames=fieldnames)` -> needs the column names upfront
- `writer.writeheader()` -> writes the header row using `fieldnames`
- `writer.writerow({...})` -> each dictionary key must match one of the `fieldnames`
- `csv.DictReader(f)` -> automatically uses the **first row as header**, and returns each row as a dictionary (`OrderedDict`/`dict`) - much easier to read by column name instead of index


In [ ]:
# Real-world Example - Simple EDA-style summary: average age from a CSV file

import csv

total_age = 0
count = 0

with open("students2.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_age += int(row["age"])   # convert string to int
        count += 1

average_age = total_age / count
print(f"Average age of {count} students: {average_age}")


**Line by line explanation:**
- We loop through every row using `DictReader`
- `int(row["age"])` -> CSV values are always read as **strings**, so we must convert them to the correct type manually
- We accumulate total and count, then compute the average
- This is a very common pattern used in early-stage EDA (Exploratory Data Analysis) before moving to Pandas


## 21. Internal Working (CSV)

- Internally, `csv.reader`/`csv.writer` handle line splitting by commas, but also correctly handle:
  - Commas inside quoted fields (e.g., `"New York, USA"` stays as one field)
  - Different line endings across operating systems
- `DictReader` internally reads the first row, stores it as the **fieldnames**, and for every subsequent row, zips values with these fieldnames to build a dictionary
- This is why `newline=""` matters on Windows - without it, the underlying file write adds extra `\r\n` characters causing blank rows


## 22. Time & Space Complexity (CSV)

- Reading with `csv.reader`/`DictReader` -> Time: O(n), Space: O(1) extra per row if processed in a loop (only one row in memory at a time), or O(n) if all rows are collected into a list
- Writing with `csv.writer`/`DictWriter` -> Time: O(n), where n is total number of rows written


## 23. Common Mistakes (CSV)

- Forgetting `newline=""` while writing on Windows -> results in extra blank lines between rows
- Forgetting that all values read from CSV are **strings** -> causes errors when doing math directly on them
- Mismatched `fieldnames` between `DictWriter` and the dictionary being written -> raises `ValueError`
- Assuming the CSV always uses commas -> some files use `;` or `\t` as separators; `csv.reader(f, delimiter=";")` handles this


## 24. Best Practices (CSV)

- Always use `newline=""` when opening files for CSV writing
- Always specify `encoding="utf-8"`
- Prefer `DictReader`/`DictWriter` for readability when column names matter
- Convert data types explicitly right after reading (e.g., `int()`, `float()`)
- For large-scale real data analysis, prefer the **Pandas** library (`pd.read_csv`) over manually looping with the `csv` module (covered in Day 4)


## 25. Modules and Imports - Introduction

- A **module** is simply a Python file (`.py`) containing reusable code - functions, classes, variables
- A **package** is a folder containing multiple related modules (with an `__init__.py` file, in older Python versions)
- Why do we need it?
  - To organize code into logical, reusable, maintainable pieces
  - To reuse code across multiple projects without rewriting it
  - Python's standard library itself is made of built-in modules (`math`, `json`, `csv`, `os`, etc.)
- Where is it used?
  - Every real Python project - separating code into files like `utils.py`, `models.py`, `helpers.py`


## 26. Real-Life Analogy

- Think of modules like **chapters in a textbook**
  - Each chapter (module) covers one topic and can be referred to independently
  - A package is like the **entire textbook/book**, organizing multiple chapters together
- `import` is like **referring to a specific chapter** instead of rewriting its content in your own notes


## 27. Explanation

- Three common ways to import in Python:
  - `import module_name` -> imports the whole module; access items using `module_name.item`
  - `from module_name import item` -> imports a specific function/class/variable directly
  - `import module_name as alias` -> imports with a shorter custom name
- `__name__ == "__main__"`:
  - Every Python file has a built-in variable `__name__`
  - If the file is run **directly**, `__name__` is set to `"__main__"`
  - If the file is **imported** into another file, `__name__` is set to the module's actual name
  - This lets us write code that only runs when the file is executed directly, not when imported


## 28. Syntax

```python
# Importing a full built-in module
import math
print(math.sqrt(16))

# Importing a specific function
from math import sqrt
print(sqrt(16))

# Importing with an alias
import math as m
print(m.sqrt(16))

# Importing your own module (a file called my_module.py in the same folder)
import my_module
my_module.some_function()

# Standard "main guard" pattern
if __name__ == "__main__":
    print("This runs only when the file is executed directly")
```


In [ ]:
# Basic Example - Using Python's built-in modules

import math
import random

print("Square root of 25:", math.sqrt(25))
print("Value of pi:", math.pi)
print("Random number between 1-10:", random.randint(1, 10))


**Line by line explanation:**
- `import math` -> loads the built-in `math` module, giving access to mathematical functions/constants
- `math.sqrt(25)` -> calls the `sqrt` function defined inside the `math` module
- `math.pi` -> a constant defined inside the module
- `import random` -> loads the built-in `random` module
- `random.randint(1, 10)` -> generates a random integer between 1 and 10 (inclusive)


In [ ]:
# Intermediate Example - Creating and using our own module

# Step 1: Create a module file called 'greetings.py'
module_code = '''
def greet(name):
    return f"Hello, {name}! Welcome to Python."

def farewell(name):
    return f"Goodbye, {name}! See you soon."

PI_APPROX = 3.14

if __name__ == "__main__":
    print("This runs only when greetings.py is executed directly, not when imported")
'''

with open("greetings.py", "w", encoding="utf-8") as f:
    f.write(module_code)

# Step 2: Import and use our own module
import greetings

print(greetings.greet("Sandipan"))
print(greetings.farewell("Sandipan"))
print("Constant from module:", greetings.PI_APPROX)


**Line by line explanation:**
- We first **write a Python file `greetings.py`** to disk using the file handling we learned earlier
- `import greetings` -> imports our own custom module (Python finds it because it is in the same folder)
- `greetings.greet("Sandipan")` -> calls the function defined inside the module
- Notice: the `if __name__ == "__main__":` block inside `greetings.py` does **not** run here, because we imported the module rather than running it directly
- `greetings.PI_APPROX` -> accessing a variable defined in the module


In [ ]:
# Real-world Example - Organizing reusable utility functions in a module

utils_code = '''
def celsius_to_fahrenheit(c):
    return (c * 9/5) + 32

def is_even(n):
    return n % 2 == 0

def clean_text(text):
    return text.strip().lower()
'''

with open("utils.py", "w", encoding="utf-8") as f:
    f.write(utils_code)

# Using specific imports with 'from ... import'
from utils import celsius_to_fahrenheit, is_even, clean_text

print(celsius_to_fahrenheit(100))
print(is_even(7))
print(clean_text("   Hello WORLD   "))


**Line by line explanation:**
- We create a `utils.py` file containing small, independent, reusable helper functions
- `from utils import celsius_to_fahrenheit, is_even, clean_text` -> imports only the specific functions we need, directly into our current namespace
- We can now call them directly without the `utils.` prefix
- This mirrors real-world projects where common helper functions are grouped into a `utils.py` file and reused across the codebase


## 29. Internal Working (Modules)

- When you write `import module_name`, Python does the following internally:
  1. Checks if the module is already loaded in `sys.modules` (a cache of already-imported modules)
  2. If not loaded, searches for the module file in the directories listed in `sys.path`
  3. Once found, Python **executes the entire module file top to bottom**, once
  4. All top-level code (function defs, class defs, variables) becomes available as attributes of the module object
  5. The module is then cached in `sys.modules`, so importing it again elsewhere does **not** re-execute it
- This is exactly why `if __name__ == "__main__":` matters - the module file's top-level code always runs once on import, but code inside this guard runs only when the file itself is the "main" program being executed


## 30. Common Mistakes (Modules)

- Naming your own file the same as a built-in module (e.g., `random.py`) -> causes conflicts and import errors
- Using `from module import *` -> imports everything, including names you didn't expect, and can cause naming clashes; avoid this in real projects
- Forgetting that importing a module only **executes it once** - changing global state after import doesn't re-run the module
- Circular imports - two modules importing each other, causing errors


## 31. Best Practices (Modules)

- Keep related functions/classes together in one meaningful module (e.g., `utils.py`, `models.py`)
- Avoid `from module import *`; import only what you need
- Use `if __name__ == "__main__":` in every script that might also be imported elsewhere
- Follow naming conventions: module names should be short, lowercase, with underscores if needed (`data_cleaning.py`, not `DataCleaning.py`)
- Group standard library imports, third-party imports, and your own module imports separately at the top of the file (PEP 8 convention)


## 32. Revision Summary

**File I/O**
- `open(file, mode, encoding)` opens a file; always prefer `with open(...) as f:` so it auto-closes
- Modes: `"r"` read, `"w"` write/overwrite, `"a"` append, add `"b"` for binary
- `read()` -> entire content as string; `readline()` -> one line; `readlines()` -> list of lines; `for line in f:` -> most memory efficient
- Internally uses buffered I/O; `with` guarantees the buffer is flushed and file closed even on error

**JSON**
- `dump`/`load` work with **files**; `dumps`/`loads` work with **strings**
- Python dict <-> JSON object, Python list <-> JSON array
- Use `indent=4` for human-readable JSON files
- Common in APIs and config files

**CSV**
- `csv.reader`/`csv.writer` work with **lists**; `DictReader`/`DictWriter` work with **dictionaries** using header row as keys
- Always use `newline=""` when writing on Windows
- All CSV values are read as strings - convert types manually
- For large-scale data work, Pandas (`pd.read_csv`) is preferred over manual CSV module (Day 4)

**Modules & Imports**
- `import module`, `from module import item`, `import module as alias`
- `if __name__ == "__main__":` runs code only when file is executed directly, not when imported
- Modules are cached in `sys.modules` after first import - only executed once
- Avoid `from module import *`; group imports (standard library, third-party, own modules) at the top

> Interview tip: A common question is "difference between `dump`/`dumps` and `load`/`loads`" and "what does `if __name__ == '__main__'` do" - make sure both are crystal clear.

---
*Day 2 (Functions & OOP) complete with this notebook. Interview Q&A (60 questions: Easy/Medium/Hard) for the full syllabus will be covered at the end, as requested.*
